In [ ]:
import pandas as pd
import numpy as np
import sqlite3

mock_data = [
    {'account_id': 1, 'mission_id': 1, 'result': 'win', 'duration_sec': 100, 'started_at': '2021-04-02', 'install_date': '2021-04-01'},
    {'account_id': 1, 'mission_id': 1, 'result': 'win', 'duration_sec': 120, 'started_at': '2021-04-02', 'install_date': '2021-04-01'},
    {'account_id': 1, 'mission_id': 1, 'result': 'win', 'duration_sec': 110, 'started_at': '2021-04-03', 'install_date': '2021-04-01'},
    {'account_id': 1, 'mission_id': 1, 'result': 'win', 'duration_sec': 130, 'started_at': '2021-04-03', 'install_date': '2021-04-01'},
    {'account_id': 1, 'mission_id': 1, 'result': 'lose', 'duration_sec': 50, 'started_at': '2021-04-03', 'install_date': '2021-04-01'},

    {'account_id': 2, 'mission_id': 1, 'result': 'win', 'duration_sec': 150, 'started_at': '2021-04-05', 'install_date': '2021-04-04'},
    {'account_id': 2, 'mission_id': 1, 'result': 'win', 'duration_sec': 160, 'started_at': '2021-04-05', 'install_date': '2021-04-04'},
    {'account_id': 2, 'mission_id': 1, 'result': 'lose', 'duration_sec': 20, 'started_at': '2021-04-06', 'install_date': '2021-04-04'},

    {'account_id': 3, 'mission_id': 1, 'result': 'win', 'duration_sec': 200, 'started_at': '2021-04-05', 'install_date': '2021-02-01'},
    {'account_id': 3, 'mission_id': 1, 'result': 'win', 'duration_sec': 200, 'started_at': '2021-04-05', 'install_date': '2021-02-01'},
    {'account_id': 3, 'mission_id': 1, 'result': 'win', 'duration_sec': 200, 'started_at': '2021-04-05', 'install_date': '2021-02-01'},
    {'account_id': 3, 'mission_id': 1, 'result': 'win', 'duration_sec': 200, 'started_at': '2021-04-05', 'install_date': '2021-02-01'},

    {'account_id': 4, 'mission_id': 1, 'result': 'lose', 'duration_sec': 10, 'started_at': '2021-04-10', 'install_date': '2021-04-10'},
    {'account_id': 4, 'mission_id': 1, 'result': 'lose', 'duration_sec': 15, 'started_at': '2021-04-10', 'install_date': '2021-04-10'},

    {'account_id': 5, 'mission_id': 2, 'result': 'win', 'duration_sec': 300, 'started_at': '2021-05-01', 'install_date': '2021-05-01'},
    {'account_id': 5, 'mission_id': 2, 'result': 'win', 'duration_sec': 300, 'started_at': '2021-05-01', 'install_date': '2021-05-01'},
    {'account_id': 5, 'mission_id': 2, 'result': 'win', 'duration_sec': 310, 'started_at': '2021-05-01', 'install_date': '2021-05-01'},
    {'account_id': 5, 'mission_id': 2, 'result': 'win', 'duration_sec': 310, 'started_at': '2021-05-01', 'install_date': '2021-05-01'}
]

df_missions = pd.DataFrame(mock_data)

conn = sqlite3.connect(':memory:')
df_missions.to_sql('missions', conn, index=False)

query = """
WITH player_mission_stats AS (
    SELECT
        mission_id,
        account_id,
        SUM(CASE WHEN result = 'win' THEN duration_sec ELSE 0 END) AS win_duration_sec,
        COUNT(CASE WHEN result = 'win' THEN 1 END) AS win_runs,
        SUM(CASE WHEN result = 'lose' THEN duration_sec ELSE 0 END) AS lose_duration_sec,
        COUNT(CASE WHEN result = 'lose' THEN 1 END) AS lose_runs
    FROM missions
    WHERE install_date >= '2021-03-01'
    GROUP BY
        mission_id,
        account_id
)
SELECT
    mission_id,
    SUM(win_duration_sec) * 1.0 / NULLIF(SUM(win_runs), 0) AS "Средняя длительность удачного прохождения",
    SUM(lose_duration_sec) * 1.0 / NULLIF(SUM(lose_runs), 0) AS "Средняя длительность неудачного прохождения",
    COUNT(CASE WHEN win_runs > 3 THEN account_id END) * 1.0 / COUNT(account_id) AS "Доля игроков с более чем 3 победами среди всех, кто запускал эту миссию"
FROM player_mission_stats
GROUP BY
    mission_id
ORDER BY
    mission_id;
"""

result_sql = pd.read_sql(query, conn)

pd.set_option('display.float_format', '{:.2f}'.format)
display(result_sql)

conn.close()

,mission_id,Средняя длительность удачного прохождения,Средняя длительность неудачного прохождения,"Доля игроков с более чем 3 победами среди всех, кто запускал эту миссию"
0,1,128.33,23.75,0.33
1,2,305.00,NaN,1.00
